In [2]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
import os

from google.adk.sessions import Session
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig
from google.genai import types


from utils.tools import check_warehouse_availability, reserve_warehouse_items

In [ ]:
model = LiteLlm(model="gpt-4.1-mini",
                temperature=0.0,
                api_key=os.getenv("OPENAI_API_KEY"),
                )


warehouse_agent = Agent(
    name="warehouse_agent",
    model=model,
    description="A warehouse agent that checks the availability of items in the warehouse and reserves them",
    tools=[check_warehouse_availability, reserve_warehouse_items],
    instruction="""
    You are a part of the shopping assistant that can manage available inventory in the warehouses.

You will be given a conversation history and a list of tools, your task is to perform actions requested by the latest user query. Answer part of the query that you can answer with the available tools.

Instructions:
- You must always check the availability of the items in the warehouses before reserving them.
- Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
- If you cannot reserve any items, return an answer that the order cannot be reserved.
- If you can reserve some items, return an answer that the order can be partially reserved and include the details.
- If only partial quantity can be reserved in some warehouses, try to combinethe required quantity from different warehouses.
- Try to reserve items from the closest warehouse to the user first if users location is provided.""")


In [4]:
from google.adk.sessions import InMemorySessionService
session_service = InMemorySessionService()
runner = Runner(
    agent=warehouse_agent,
    app_name="warehouse_agent",
    session_service=session_service)

session = await session_service.create_session(
        app_name="warehouse_agent",
        user_id="krishnak",
        session_id="session_123",
        )
    

In [5]:
user_msg = types.Content(role="user", 
                         parts=[types.Part(text="What is the availability of B09WCL37Z4 in all of the warehouses?")])


In [9]:
run_config = RunConfig(
    max_llm_calls=3
    )
response = runner.run(user_id="krishnak", session_id=session.id, new_message=user_msg, run_config=run_config)


In [11]:
for event in response:
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(part.text)

The product B09WCL37Z4 is available in all the warehouses with the following quantities:
- Mumbai Logistics Hub (Mumbai, India): 59 units available
- Chennai Regional Warehouse (Chennai, India): 91 units available
- Kolkata Eastern Hub (Kolkata, India): 54 units available

Let me know if you want to reserve any quantity from these warehouses.


In [17]:
APP_NAME = "warehouse_agent"
_runner = None
_session_service = None

async def ask(query: str, session_id: str = "default", user_id: str = "user1") -> str:
    """Send a query to the agent and return the final text response."""
    
    global _runner, _session_service

    if _runner is None:
        session_service = InMemorySessionService()
        runner = Runner(
            agent=warehouse_agent,
            app_name=APP_NAME,
            session_service=session_service)
    
    # Create session if it doesn't exist (idempotent)
    session = await session_service.get_session(
        app_name=APP_NAME, user_id=user_id, session_id=session_id
    )
    if session is None:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=user_id, session_id=session_id
        )

    user_msg = types.Content(
        role="user",
        parts=[types.Part.from_text(text=query)]
    )

    response = runner.run_async(
        user_id=user_id, session_id=session.id, new_message=user_msg
    )

    # Collect final text from streamed events
    final_text = ""
    async for event in response:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    final_text += part.text

    return final_text

In [21]:
print(await ask("What is the availability of B09WCL37Z4 in Chennai warehouses?", session_id="chat-1"))


The product B09WCL37Z4 is available in the Chennai Regional Warehouse with a quantity of 91 units. It is fully available for reservation there. Would you like to reserve any quantity of this product from the Chennai warehouse?
